# Local Unified Candidate Inspector

Run this notebook from the root of your local repository:

```text
kaggle-cohort-x-task-1-solution/
├── unified_candidates/
├── parsed/
├── retrieval.py
└── ...
```

It does not import any project helper functions.


In [1]:
from pathlib import Path
from typing import Any
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()

# If needed, replace with an absolute path:
# REPO_ROOT = Path("/absolute/path/to/kaggle-cohort-x-task-1-solution")

UNIFIED_CANDIDATE_DIR = REPO_ROOT / "unified_candidates"
PARSED_DIR = REPO_ROOT / "parsed"
OUTPUT_DIR = REPO_ROOT / "candidate_diagnostics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Unified candidates:", UNIFIED_CANDIDATE_DIR)
print("Parsed directory:", PARSED_DIR)
print("Output directory:", OUTPUT_DIR)

assert UNIFIED_CANDIDATE_DIR.exists(), (
    f"Directory not found: {UNIFIED_CANDIDATE_DIR}"
)


Repository root: /Users/neilchen/Desktop/Research & Projects/ucsf_agent/kaggle-cohort-x-task-1-solution
Unified candidates: /Users/neilchen/Desktop/Research & Projects/ucsf_agent/kaggle-cohort-x-task-1-solution/unified_candidates
Parsed directory: /Users/neilchen/Desktop/Research & Projects/ucsf_agent/kaggle-cohort-x-task-1-solution/parsed
Output directory: /Users/neilchen/Desktop/Research & Projects/ucsf_agent/kaggle-cohort-x-task-1-solution/candidate_diagnostics


In [2]:
TEXT_KEYS = ["text", "content", "paragraph", "block_text"]
SECTION_KEYS = ["section_path", "path", "section", "heading"]
ID_KEYS = ["block_id", "id", "unit_id", "candidate_id"]
SCORE_KEYS = ["score", "retrieval_score", "similarity", "relevance_score"]

def clean_text(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

def first_present(item: dict, keys: list[str], default=None):
    for key in keys:
        if key in item and item[key] not in [None, ""]:
            return item[key]
    return default

def find_candidate_list(payload: Any) -> list:
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return []

    for key in [
        "candidates",
        "blocks",
        "units",
        "items",
        "retrieved_blocks",
        "unified_candidates",
    ]:
        value = payload.get(key)
        if isinstance(value, list):
            return value

    for value in payload.values():
        if isinstance(value, dict):
            nested = find_candidate_list(value)
            if nested:
                return nested

    return []


In [3]:
def locate_candidate_file(pmcid: str) -> Path:
    pmcid = str(pmcid).strip().replace("PMC", "")

    for path in [
        UNIFIED_CANDIDATE_DIR / f"PMC{pmcid}.json",
        UNIFIED_CANDIDATE_DIR / f"{pmcid}.json",
    ]:
        if path.exists():
            return path

    matches = sorted(
        UNIFIED_CANDIDATE_DIR.rglob(f"*{pmcid}*.json")
    )
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"No unified candidate JSON found for PMC{pmcid}"
    )

def load_raw_candidate_payload(pmcid: str):
    path = locate_candidate_file(pmcid)
    with open(path, encoding="utf-8") as file:
        payload = json.load(file)
    return path, payload

def normalize_candidate_item(item: Any, index: int) -> dict | None:
    if isinstance(item, str):
        text = clean_text(item)
        if not text:
            return None
        return {
            "candidate_index": index,
            "block_id": str(index),
            "section_path": "",
            "retrieval_score": np.nan,
            "character_count": len(text),
            "word_count": len(text.split()),
            "text": text,
            "raw_keys": [],
        }

    if not isinstance(item, dict):
        return None

    text = clean_text(first_present(item, TEXT_KEYS, ""))
    if not text:
        return None

    section_path = clean_text(
        first_present(item, SECTION_KEYS, "")
    )

    return {
        "candidate_index": index,
        "block_id": str(first_present(item, ID_KEYS, index)),
        "section_path": section_path,
        "retrieval_score": first_present(
            item, SCORE_KEYS, np.nan
        ),
        "character_count": len(text),
        "word_count": len(text.split()),
        "text": text,
        "raw_keys": sorted(item.keys()),
    }


## Inspect one paper

In [9]:
PMCID = "10542709"

candidate_path, raw_payload = load_raw_candidate_payload(PMCID)

print("File:", candidate_path)
print("Top-level type:", type(raw_payload).__name__)

if isinstance(raw_payload, dict):
    print("Top-level keys:", list(raw_payload.keys()))

raw_json_text = json.dumps(
    raw_payload,
    indent=2,
    ensure_ascii=False,
)

print(raw_json_text[:30000])


File: /Users/neilchen/Desktop/Research & Projects/ucsf_agent/kaggle-cohort-x-task-1-solution/unified_candidates/10542709.json
Top-level type: dict
Top-level keys: ['doc_id', 'title', 'abstract', 'keywords', 'num_blocks', 'num_candidates', 'total_chars', 'target_chars', 'selected_chars', 'selected_char_fraction', 'retrieval_config', 'queries', 'candidates', 'top_score', 'lowest_selected_score']
{
  "doc_id": "PMC10542709",
  "title": "Atorvastatin lowers 68 Ga-DOTATATE uptake in coronary arteries, bone marrow and spleen in individuals with type 2 diabetes",
  "abstract": "Aims/hypothesis Inflammation is a core component of residual cardiovascular risk in type 2 diabetes. With new anti-inflammatory therapeutics entering the field, accurate markers to evaluate their effectiveness in reducing cardiovascular disease are paramount. Gallium-68-labelled DOTATATE ( 68 Ga-DOTATATE) has recently been proposed as a more specific marker of arterial wall inflammation than 18 F-fluorodeoxyglucose ( 1

In [10]:
raw_items = find_candidate_list(raw_payload)

rows = []
for index, item in enumerate(raw_items):
    row = normalize_candidate_item(item, index)
    if row is not None:
        rows.append(row)

candidate_df = pd.DataFrame(rows)

print("Raw items:", len(raw_items))
print("Usable candidates:", len(candidate_df))

display(candidate_df)


Raw items: 8
Usable candidates: 8


,candidate_index,block_id,section_path,retrieval_score,character_count,word_count,text,raw_keys
0,0,PMC10542709::block_0,['Introduction'],0.538153,1471,211,Type 2 diabetes is hallmarked by systemic infl...,"[block_id, block_index, block_type, char_count..."
1,1,PMC10542709::block_2,['Methods'],0.602545,1262,203,"In short, individuals with type 2 diabetes fro...","[block_id, block_index, block_type, char_count..."
2,2,PMC10542709::block_3,"['Results', 'Patient characteristics']",0.623505,3082,508,"Of the 24 patients included, one patient withd...","[block_id, block_index, block_type, char_count..."
3,3,PMC10542709::block_4,"['Results', 'Changes in 68 Ga-DOTATATE uptake ...",0.546239,1000,165,The imaging parameters at baseline and follow-...,"[block_id, block_index, block_type, char_count..."
4,4,PMC10542709::block_5,['Discussion'],0.552115,899,128,We provide evidence of therapeutic modulation ...,"[block_id, block_index, block_type, char_count..."
5,5,PMC10542709::block_7,['Discussion'],0.542357,1206,183,We observed a notably higher uptake of 68 Ga-D...,"[block_id, block_index, block_type, char_count..."
6,6,PMC10542709::block_8,['Discussion'],0.574473,1018,156,Limitations of our study include the lack of a...,"[block_id, block_index, block_type, char_count..."
7,7,PMC10542709::block_10,"['Untitled Section', 'Supplementary Information']",0.552030,92,14,Below is the link to the electronic supplement...,"[block_id, block_index, block_type, char_count..."


In [6]:
for _, row in candidate_df.iterrows():
    print("\n" + "=" * 120)
    print("Candidate index:", row["candidate_index"])
    print("Block ID:", row["block_id"])
    print("Section:", row["section_path"])
    print("Retrieval score:", row["retrieval_score"])
    print("Characters:", row["character_count"])
    print("Words:", row["word_count"])
    print("-" * 120)
    print(row["text"])



Candidate index: 0
Block ID: PMC10542709::block_3
Section: ['Results', 'Patient characteristics']
Retrieval score: 0.6235047578811646
Characters: 3082
Words: 508
------------------------------------------------------------------------------------------------------------------------
Of the 24 patients included, one patient withdrew from the study prior to first scan and another patient discontinued study participation owing to myalgia and did not complete follow-up PET/CT. Accordingly, 22 patients (mean age 63±6 years, 82% male, HbA 1c 55 mmol/mol [7.2%]) were included in subsequent analyses. A flowchart of the inclusions can be found in ESM Fig. 1 . The baseline and follow-up characteristics after 12 weeks of statin therapy are listed in Table 1 . Of note, LDL-cholesterol levels decreased by 58% and C-reactive protein (CRP) by 20%, while the HbA 1c did not decrease at follow-up. Table 1 Baseline characteristics and changes in laboratory variables and imaging parameters after 12 weeks 

## Summarize every file

In [ ]:
all_json_files = sorted(
    UNIFIED_CANDIDATE_DIR.rglob("*.json")
)

summary_rows = []

for path in all_json_files:
    try:
        with open(path, encoding="utf-8") as file:
            payload = json.load(file)

        items = find_candidate_list(payload)

        usable_count = 0
        total_characters = 0
        total_words = 0
        sections = set()
        numeric_scores = []

        for index, item in enumerate(items):
            row = normalize_candidate_item(item, index)
            if row is None:
                continue

            usable_count += 1
            total_characters += row["character_count"]
            total_words += row["word_count"]

            if row["section_path"]:
                sections.add(row["section_path"])

            score = pd.to_numeric(
                row["retrieval_score"],
                errors="coerce",
            )
            if pd.notna(score):
                numeric_scores.append(float(score))

        match = re.search(
            r"(?:PMC)?(\d+)",
            path.stem,
            flags=re.I,
        )
        pmcid = match.group(1) if match else path.stem

        summary_rows.append({
            "pmcid": pmcid,
            "relative_path": str(
                path.relative_to(UNIFIED_CANDIDATE_DIR)
            ),
            "candidate_count": usable_count,
            "candidate_characters": total_characters,
            "candidate_words": total_words,
            "unique_section_count": len(sections),
            "maximum_retrieval_score": (
                max(numeric_scores)
                if numeric_scores
                else np.nan
            ),
            "minimum_retrieval_score": (
                min(numeric_scores)
                if numeric_scores
                else np.nan
            ),
            "mean_retrieval_score": (
                float(np.mean(numeric_scores))
                if numeric_scores
                else np.nan
            ),
            "status": "ok",
        })

    except Exception as error:
        summary_rows.append({
            "pmcid": path.stem,
            "relative_path": str(path),
            "candidate_count": np.nan,
            "candidate_characters": np.nan,
            "candidate_words": np.nan,
            "unique_section_count": np.nan,
            "maximum_retrieval_score": np.nan,
            "minimum_retrieval_score": np.nan,
            "mean_retrieval_score": np.nan,
            "status": f"error: {error}",
        })

candidate_summary_df = pd.DataFrame(summary_rows)

display(
    candidate_summary_df.sort_values(
        ["candidate_count", "candidate_characters"],
        ascending=True,
        na_position="first",
    )
)


In [ ]:
MIN_CANDIDATE_COUNT = 5
MIN_CANDIDATE_CHARACTERS = 2000
MIN_UNIQUE_SECTIONS = 2

suspicious_df = candidate_summary_df[
    (candidate_summary_df["candidate_count"] < MIN_CANDIDATE_COUNT)
    | (
        candidate_summary_df["candidate_characters"]
        < MIN_CANDIDATE_CHARACTERS
    )
    | (
        candidate_summary_df["unique_section_count"]
        < MIN_UNIQUE_SECTIONS
    )
    | (candidate_summary_df["status"] != "ok")
].sort_values(
    [
        "candidate_count",
        "candidate_characters",
        "unique_section_count",
    ],
    ascending=True,
    na_position="first",
)

print("Suspicious papers:", len(suspicious_df))
display(suspicious_df)


In [ ]:
display(
    candidate_summary_df[
        [
            "candidate_count",
            "candidate_characters",
            "candidate_words",
            "unique_section_count",
        ]
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

valid_counts = pd.to_numeric(
    candidate_summary_df["candidate_count"],
    errors="coerce",
).dropna()

plt.figure(figsize=(9, 5))
plt.hist(
    valid_counts,
    bins=min(40, max(5, int(valid_counts.nunique()))),
)
plt.xlabel("Unified candidates per paper")
plt.ylabel("Number of papers")
plt.title("Unified candidate-count distribution")
plt.show()


## Print any chosen papers

In [ ]:
def print_candidate_files(pmcids: list[str]):
    for pmcid in pmcids:
        print("\n" + "#" * 120)
        print(f"PMC{pmcid}")
        print("#" * 120)

        try:
            path, payload = load_raw_candidate_payload(pmcid)
            items = find_candidate_list(payload)

            print("File:", path)
            print("Raw candidate items:", len(items))

            for index, item in enumerate(items):
                row = normalize_candidate_item(item, index)
                if row is None:
                    continue

                print("\n" + "=" * 100)
                print(
                    f"Candidate {index} | "
                    f"Section: {row['section_path']} | "
                    f"Score: {row['retrieval_score']}"
                )
                print("-" * 100)
                print(row["text"])

        except Exception as error:
            print("ERROR:", error)

# Example:
# print_candidate_files(["10542709", "10656797"])


## Optional: inspect parsed directory

In [ ]:
parsed_files = sorted(
    path
    for path in PARSED_DIR.rglob("*")
    if path.is_file()
)

print("Parsed files:", len(parsed_files))

for path in parsed_files[:30]:
    print(path.relative_to(REPO_ROOT))


## Save diagnostics

In [ ]:
summary_path = (
    OUTPUT_DIR
    / "unified_candidate_pool_diagnostics.csv"
)

suspicious_path = (
    OUTPUT_DIR
    / "suspicious_unified_candidate_pools.csv"
)

candidate_summary_df.to_csv(
    summary_path,
    index=False,
)

suspicious_df.to_csv(
    suspicious_path,
    index=False,
)

print("Saved:", summary_path)
print("Saved:", suspicious_path)
